[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/02-data-science-stack/03_pandas_wrangling_and_missing_data.ipynb)

# One Dirty Table, All the Way Through

**Session 5 · companion to HW 1 · nothing here is a homework answer**

Session 5 covers nine topics. This notebook covers three of them properly, on
one table, because the skill being trained is not "call `groupby`" — it is
noticing what a table is doing to you before a model does it for you:

1. **a missing-data audit** that says how much is missing and where,
2. **`agg` versus `transform`**, which differ in the shape they return,
3. **a merge that fails loudly** rather than quietly duplicating rows,

and it ends with the smallest useful piece of engineering in the whole course:
turning a cleaning rule into a test that runs.

In [1]:
import sys

import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 240

sites = rng.choice(["north", "south", "ridge"], size=n, p=[0.45, 0.35, 0.20])
readings = pd.DataFrame({
    "site": sites,
    "measured_at": pd.to_datetime("2026-06-01") + pd.to_timedelta(rng.integers(0, 90, n), "D"),
    "sensor": rng.choice(["a", "b", "c"], size=n),
    "temp_c": np.round(rng.normal(18, 4, n) + (sites == "ridge") * -6, 1),
    "humidity": np.round(rng.normal(60, 12, n), 1),
})

# Missing not at random: the ridge sensor loses readings when it is cold.
cold_ridge = (readings.site == "ridge") & (readings.temp_c < 10)
readings.loc[readings.sample(frac=0.06, random_state=1).index, "humidity"] = np.nan
readings.loc[cold_ridge.sample(frac=1.0, random_state=2)[lambda s: s].index, "temp_c"] = np.nan

readings.head()

,site,measured_at,sensor,temp_c,humidity
0,south,2026-08-20,c,20.3,57.7
1,ridge,2026-07-05,a,NaN,65.2
2,south,2026-06-15,a,19.7,68.2
3,north,2026-06-23,b,14.1,55.9
4,north,2026-08-19,c,13.2,NaN


## 1. What is missing, and does the missingness mean anything?

The audit is two lines. The thinking is what takes time: a column that is 4%
missing at random is an inconvenience, and a column that is missing *because of
its own value* is a different dataset than you think you have.

In [2]:
audit = pd.DataFrame({
    "missing": readings.isna().sum(),
    "percent": (readings.isna().mean() * 100).round(1),
    "dtype": readings.dtypes.astype(str),
})
print(audit.to_string())

             missing  percent           dtype
site               0      0.0          object
measured_at        0      0.0  datetime64[ns]
sensor             0      0.0          object
temp_c            21      8.8         float64
humidity          14      5.8         float64


In [3]:
# Is temp_c missing at random? Compare the sites, then compare what is left.
by_site = readings.assign(temp_missing=readings.temp_c.isna()).groupby("site")
print(by_site.temp_missing.mean().round(3).to_string(), "\n")

print("mean temp of the ROWS THAT SURVIVED, by site:")
print(readings.groupby("site").temp_c.mean().round(2).to_string())

site
north    0.000
ridge    0.382
south    0.000 

mean temp of the ROWS THAT SURVIVED, by site:
site
north    17.61
ridge    12.98
south    18.50


The missingness is not spread evenly: it is concentrated on `ridge`, and it is
exactly the cold readings that vanished. Drop those rows and the ridge site
looks warmer than it is — the average of what survived is not the average of
what happened.

This is Rubin's taxonomy made concrete. You cannot *prove* MAR or MNAR from the
data alone, but you can show that the missingness is related to something you
observed, and that is enough to stop you from calling `dropna()` and moving on.

In [4]:
naive = readings.dropna(subset=["temp_c"]).groupby("site").temp_c.mean()
print("after dropna, ridge reads:", round(naive["ridge"], 2), "degrees")
print("rows lost:", readings.temp_c.isna().sum(), "of", len(readings))

after dropna, ridge reads: 12.98 degrees
rows lost: 21 of 240


## 2. Group-wise imputation, and the shape question

Filling a hole with the column mean ignores the structure you just found: a
missing ridge reading should be filled with what ridge readings look like, not
with what the whole valley looks like.

`transform` is what makes that one line — and it is the first place the
`agg`/`transform` distinction actually bites.

In [5]:
agg_result = readings.groupby("site").temp_c.mean()
transform_result = readings.groupby("site").temp_c.transform("mean")

print("agg       ->", type(agg_result).__name__, agg_result.shape, "one row per group")
print("transform ->", type(transform_result).__name__, transform_result.shape, "one row per ORIGINAL row")
print()
print(pd.DataFrame({"site": readings.site, "temp_c": readings.temp_c,
                    "group_mean": transform_result}).head(4).to_string(index=False))

agg       -> Series (3,) one row per group
transform -> Series (240,) one row per ORIGINAL row

 site  temp_c  group_mean
south    20.3   18.497590
ridge     NaN   12.976471
south    19.7   18.497590
north    14.1   17.609804


In [6]:
filled = readings.copy()
filled["temp_c"] = filled.temp_c.fillna(transform_result)

print("missing before:", readings.temp_c.isna().sum(), " after:", filled.temp_c.isna().sum())
print()
print("ridge mean — dropna vs group-filled:",
      round(naive["ridge"], 2), "vs", round(filled.groupby("site").temp_c.mean()["ridge"], 2))

missing before: 21  after: 0

ridge mean — dropna vs group-filled: 12.98 vs 12.98


The two numbers are **identical**, and that is not a coincidence: filling a
group's holes with that group's own mean cannot move that group's mean. Read
that line again, because it is the point of the exercise. Imputation bought a
complete column — every downstream `sklearn` call will now run — and it bought
exactly nothing in accuracy. The cold readings are still missing from the
estimate, and the ridge still looks six degrees warmer than it was.

There is no pandas call that recovers a value nobody measured. The honest
options are to model the missingness, or to report the estimate with the caveat
attached. What you must not do is let `dropna()` — or a tidy-looking
`fillna` — make that decision silently on your behalf.

**A third shape, for completeness:** `filter` keeps or drops whole groups.

In [7]:
print("group sizes:", readings.groupby("site").size().to_dict())
big = readings.groupby("site").filter(lambda g: len(g) > 60)
print("after filter(len > 60):", big.site.unique(), "->", len(big), "rows")

group sizes: {'north': 102, 'ridge': 55, 'south': 83}
after filter(len > 60): ['south' 'north'] -> 185 rows


| Method | Returns | Use it when |
|---|---|---|
| `agg` | one row per group | you want a summary table |
| `transform` | one row per original row | you want to put the summary *back* on the rows |
| `filter` | a subset of the original rows | you want to keep or drop whole groups |

## 3. A merge that fails instead of lying

A join with a duplicated key on the right silently multiplies your rows. The
model then trains on some observations several times and you find out at the
worst possible moment — or never.

`validate=` turns that silence into an exception.

In [8]:
sites_meta = pd.DataFrame({
    "site": ["north", "south", "ridge", "ridge"],     # ridge listed twice: the bug
    "elevation_m": [640, 590, 1980, 1985],
})

sloppy = readings.merge(sites_meta, on="site", how="left")
print("rows before:", len(readings), " after a careless merge:", len(sloppy))
print("ridge rows:", (readings.site == 'ridge').sum(), "->", (sloppy.site == 'ridge').sum())

rows before: 240  after a careless merge: 295
ridge rows: 55 -> 110


In [9]:
try:
    readings.merge(sites_meta, on="site", how="left", validate="many_to_one")
except pd.errors.MergeError as err:
    print("MergeError:", err)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


That is the entire technique: state the cardinality you believe in, and let
pandas contradict you. `many_to_one`, `one_to_one`, `one_to_many` — one keyword,
and a class of silent corruption becomes a stack trace.

The fix is upstream, in the metadata table, where the duplicate never should
have been:

In [10]:
fixed_meta = sites_meta.drop_duplicates(subset="site", keep="first")
joined = readings.merge(fixed_meta, on="site", how="left", validate="many_to_one")
print("rows:", len(joined), "  unmatched sites:", joined.elevation_m.isna().sum())
print(joined.groupby("site").elevation_m.first().to_string())

rows: 240   unmatched sites: 0
site
north     640
ridge    1980
south     590


## 4. Make the rule executable

Everything above was a decision: fill within site, never drop silently, one row
in equals one row out of a left join. Decisions written in prose rot. Decisions
written as tests fail out loud when someone — usually you, in three weeks —
breaks them.

The test below is not decoration; it is the smallest version of what
`instructor/scripts/` does for this whole course.

In [11]:
import subprocess
from pathlib import Path

Path("test_cleaning_rules.py").write_text('''
import numpy as np
import pandas as pd


def clean(df, meta):
    # The cleaning contract as one function: fill within site, then join safely.
    out = df.copy()
    out["temp_c"] = out.temp_c.fillna(out.groupby("site").temp_c.transform("mean"))
    meta = meta.drop_duplicates(subset="site", keep="first")
    return out.merge(meta, on="site", how="left", validate="many_to_one")


def test_row_count_is_preserved():
    df = pd.DataFrame({"site": ["a", "a", "b"], "temp_c": [1.0, np.nan, 3.0]})
    meta = pd.DataFrame({"site": ["a", "b", "b"], "elevation_m": [10, 20, 21]})
    assert len(clean(df, meta)) == len(df)


def test_gaps_are_filled_from_the_same_site():
    df = pd.DataFrame({"site": ["a", "a", "b"], "temp_c": [1.0, np.nan, 30.0]})
    meta = pd.DataFrame({"site": ["a", "b"], "elevation_m": [10, 20]})
    assert clean(df, meta).temp_c.tolist() == [1.0, 1.0, 30.0]


def test_input_is_not_mutated():
    df = pd.DataFrame({"site": ["a", "a"], "temp_c": [1.0, np.nan]})
    meta = pd.DataFrame({"site": ["a"], "elevation_m": [10]})
    clean(df, meta)
    assert df.temp_c.isna().sum() == 1
''')

run = subprocess.run([sys.executable, "-m", "pytest", "test_cleaning_rules.py", "-q"],
                     capture_output=True, text=True)
print(run.stdout[-500:])
Path("test_cleaning_rules.py").unlink()

...                                                                      [100%]
3 passed in 0.31s



## The columnar alternatives, one query each

Polars and DuckDB are both in `requirements.txt`, so these run. Neither is
needed for any assignment; the reason to meet them now is that both read the
pandas object in place, neither will hand you a copy-versus-view surprise, and
both keep working when the table stops fitting in memory.

In [12]:
import polars as pl

same_fill = (pl.from_pandas(readings)
             .with_columns(pl.col("temp_c")
                           .fill_null(pl.col("temp_c").mean().over("site"))
                           .alias("temp_filled")))

print("polars", pl.__version__)
print(same_fill.select(["site", "temp_c", "temp_filled"]).head(4))
print("nulls left:", same_fill.get_column("temp_filled").null_count())

polars 1.43.2
shape: (4, 3)
┌───────┬────────┬─────────────┐
│ site  ┆ temp_c ┆ temp_filled │
│ ---   ┆ ---    ┆ ---         │
│ str   ┆ f64    ┆ f64         │
╞═══════╪════════╪═════════════╡
│ south ┆ 20.3   ┆ 20.3        │
│ ridge ┆ null   ┆ 12.976471   │
│ south ┆ 19.7   ┆ 19.7        │
│ north ┆ 14.1   ┆ 14.1        │
└───────┴────────┴─────────────┘
nulls left: 0


In [13]:
import duckdb

print("duckdb", duckdb.__version__)
print(duckdb.sql('''
    SELECT site,
           COUNT(*)                                        AS rows,
           ROUND(AVG(temp_c), 2)                           AS mean_observed,
           SUM(CASE WHEN temp_c IS NULL THEN 1 ELSE 0 END) AS missing
    FROM readings
    GROUP BY site
    ORDER BY site
'''))

duckdb 1.5.5
┌─────────┬───────┬───────────────┬─────────┐
│  site   │ rows  │ mean_observed │ missing │
│ varchar │ int64 │    double     │ int128  │
├─────────┼───────┼───────────────┼─────────┤
│ north   │   102 │         17.61 │       0 │
│ ridge   │    55 │         12.98 │      21 │
│ south   │    83 │          18.5 │       0 │
└─────────┴───────┴───────────────┴─────────┘



DuckDB read the pandas DataFrame straight out of the local namespace — no load
step, no copy. The SQL is the same group-by you wrote above and it reports the
same missingness, which is the point: these are different syntaxes for one set
of ideas, not different ideas.

## What to take from this

- Audit missingness **by group** before dropping anything. A column that goes
  missing for a reason is not a column you can `dropna()`.
- `agg` summarises, `transform` broadcasts back, `filter` keeps whole groups.
  Getting the shape wrong is the most common groupby bug.
- Say what a join should do with `validate=` and let pandas prove you wrong.
- A cleaning rule that is not a test is a rumour.

## Where to go next

- **Reading, Session 5** — tidy data, Rubin's taxonomy, the four join flavors.
- **HW 1** — the functions it grades operate on tables like this one. None of
  them is implemented above.